In [ ]:
import os
import numpy as np
import math
import matplotlib.pyplot as plt
import tensorflow as tf
import librosa

from scipy import ndimage
from tensorflow import keras

from keras._tf_keras.keras.utils import to_categorical
from keras._tf_keras.keras.layers import (
     Dense
)
from keras._tf_keras.keras.applications import MobileNetV2
from keras._tf_keras.keras.applications.mobilenet import preprocess_input

from sklearn.model_selection import train_test_split

from keras._tf_keras.keras.layers import GlobalAveragePooling2D
from keras._tf_keras.keras.optimizers import Adam
from keras._tf_keras.keras.losses import CategoricalCrossentropy
from keras._tf_keras.keras.layers import Dropout
from keras._tf_keras.keras.callbacks import ModelCheckpoint

2025-12-22 13:55:11.555498: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-22 13:55:11.587178: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-22 13:55:11.587218: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-22 13:55:11.605951: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-22 13:55:12.704652: W tensorflow/compiler/tf

In [2]:
print(tf.__version__)

2.16.2


## Load GTZAN dataset

#### Implementation comments

From what I read one option to process the audio samples of each genre is to use spectrograms.

A spectogram is a visual representation of the spectrum of frequencies of a signal as it varies with time (time on the x axis and frequency on the y axis). It is generated by analyzing a signal in short time segments ofthen using the Short-Time Fourier Transform.

To be able to use the MobileNetV2 pre-trained model It is required to obtain a spectrogram of each audio sample. In order to achieve this one option is to use a Mel Spectrogram.

A Mel Spectrogram is a spectrogram where the frequency axis is transformed to the Mel scale, a perceptual scale that approximates the non-linear way the human ear perceives sound. It uses a logarithmic scale that reflects human auditory sensitivity, making it more aligned with how
humans perceive pitch and loudness. This is achieved by applying a bank of overlapping triangular filters, known as a Mel filterbank, to the power spectrum obtained from a Short-Time Fourier Transform.

In order to achieve this I came across a package called librosa which is a python package designed for audio and music analysis used to process and extract maningful information from audio signals.

#### Useful documentation

  * [mobilenet docs](https://keras.io/api/applications/mobilenet/)

  * [librosa docs](https://librosa.org/doc/latest/tutorial.html)

Some considerations for the mel spectrogram:
  * *Sample rate*: 16,000 Hz for balanced quality and computational efficiency, up to 22,050 for higher fidelity.
  * *Window size*: 25 ms which corresponds to 400 samples at 16,000 Hz
  * *Window offset* (hop length): 8.33 ms approximately 1/3 of the window size, provides 75% overlap between frames, effective to capture temporal dynamics.
  * *Number of mel filters*: Commonly between 40-80 for meaningful frequency bands, up to 128 for higher resolution.
  * *Frequency range*: minimum frequency is typically set to the lowest perceptually relevant frequency 0 or 0.1 Hz, for maximum frequency it is capped to 8,000 Hz to match human hearing limits.
  * *FFT length*: Should be chosen to balance frequency resolution and computational cost, a length of 2048 or 4096 is common, 2048 is sufficient for most applications, a power of 2 is recommended
  * *Normalization*: Helps stabilizing training in neural networks
  * *Logarithmic Scaling*: Converting the power spectrogram to decibels

In [3]:
#path to dataset
GTZAN_DS = "../data/gtzan_ds/"
#genres in alphanumeric order (same as in gtzan_ds folder)
GENRES = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock"
]
#audio sample extension
WAV=".wav"

#path to store spectrogram np array values
GTZAN_NP_ARR="../data/gtzan_np_arr/"
#arr extension
NPY="npy"

In [4]:
#create folders to store spectrogram images
for genre in GENRES:
    os.makedirs(os.path.join(GTZAN_NP_ARR, genre), exist_ok=True)

In [5]:
def nearest_power_of_2(n):
    if n <= 0:
        raise ValueError("Number must be positive")
    k = math.floor(math.log2(n))
    lower_power = 2 ** k
    upper_power = 2 ** (k + 1)
    return lower_power if (n - lower_power) <= (upper_power - n) else upper_power

In [6]:
#constants from previous considerations
MS=1000 #to convert to miliseconds
OFF_SZ=3    #value for window offset size

DUR=30
SMPL_RT=16_000  # for sampling rate
WIN_TM=128   # value for window size later converted to miliseconds
WIN_SZ=nearest_power_of_2((WIN_TM/MS)*SMPL_RT)  #window size, recommended to use a power of 2
WIN_OFF=nearest_power_of_2(((WIN_TM/OFF_SZ)/MS)*SMPL_RT)    # window offset, recommended to use a power of 2
MEL_BANDS=128   #number of mel bands 40-80 up to 128
MIN_FRQ=0   #minimum frequency value
MAX_FRQ=8_000   #maximum frequency value
PWR=2   #power spectrogram value, 2 for power spectrogram

IMG_SZ=224    #for imagenetv2 image size

In [7]:
#function to create mel spectrogram from audio sample using librosa
def to_mel_spectrogram(
        source, 
        duration=DUR, 
        sample_rate=SMPL_RT,
        window_size=WIN_SZ,
        window_offset=WIN_OFF,
        mel_bands=MEL_BANDS,
        min_frq=MIN_FRQ,
        max_frq=MAX_FRQ,
        power=PWR,
        img_size=IMG_SZ
    ):
    #load audio sample
    audio_waveform, sampling_rate = librosa.load(
        source,
        duration=duration,
        sr=sample_rate
    )
    #create spectrogram from audio sample
    spectrogram = librosa.feature.melspectrogram(
        y=audio_waveform,
        sr=sampling_rate,
        n_fft=window_size,
        hop_length=window_offset,
        n_mels=mel_bands,
        fmin=min_frq,
        fmax=max_frq,
        power=power
    )
    #scale amplitude relative to max value in spectrogram
    #i.o.w highest power value in spectrogram is teated as 0 dB
    log_spectrogram = librosa.power_to_db(spectrogram, ref=np.max)
    #convert to rgb img, shape goes from (Height, Width) to (Height, Width, 3)
    #this starts to look familiar (H,W,C)
    rgb_log_spectrogram = np.stack([log_spectrogram] * 3, axis=-1)
    #resize to desired size
    resized_rgb_log_spectrogram = ndimage.zoom(
        rgb_log_spectrogram,
        (
            img_size/rgb_log_spectrogram.shape[0],    #scale height to 224
            img_size/rgb_log_spectrogram.shape[1],    #scale width to 224
            1   #keep color channel unchanged
        ),
        order=1 #order 0 avoids interpolation, using 1 balances quality and speed
    )

    return resized_rgb_log_spectrogram
    

In [8]:
#function to create spectrograms and save img and store spectrogram arrays
def generate_save_spectrograms(source_folder, classes, arr_folder, save_arr=False, load=False):
    spectrograms, genres = [], []
    if not load:
        #create spectrograms
        for idx, genre in enumerate(classes):
            genre_folder = os.path.join(source_folder, genre) # i.e. ../data/gtzan_ds/blues
            for audio in os.listdir(genre_folder)[:100]:   #each genre folder has 100 audio samples
                if audio.endswith(WAV):
                    audio_sample = os.path.join(genre_folder, audio)
                    try:
                        if save_arr:
                            #save spectrogram arr to respective np arr genre folder
                            arr_name = audio[:-3]
                            arr_name = arr_name+NPY
                            save_arr_to = os.path.join(arr_folder, genre, arr_name)
                            
                            spectrogram_arr = to_mel_spectrogram(audio_sample)
                            np.save(save_arr_to, spectrogram_arr)

                        #store spectrogram array to feature matrix
                        spectrograms.append(spectrogram_arr)
                        genres.append(idx)
                    except Exception as e:
                        print(f"Error processing audio sample: {audio} \n error: {e}")
                        continue
    else:
        #load spectrogram arrays
        for idx, genre in enumerate(classes):
            genre_folder = os.path.join(arr_folder, genre) # i.e. ../data/gtzan_np_arr/blues
            for arr in os.listdir(genre_folder)[:100]:   #each genre folder has 100 arrays except jazz 99
                if arr.endswith(NPY):
                    spectrogram = os.path.join(genre_folder, arr)
                    try:
                        spectrogram_arr = np.load(spectrogram)
                        #store spectrogram array to feature matrix
                        spectrograms.append(spectrogram_arr)
                        genres.append(idx)
                    except Exception as e:
                        print(f"Error processing spectrogram array: {arr} \n error: {e}")
                        continue
    
    return spectrograms, genres

In [9]:
#create spectrograms and save spectrogram arrays only execute once
#X, y = generate_save_spectrograms(GTZAN_DS, GENRES, GTZAN_NP_ARR, save_arr=True, load=False)
#X = np.array(X)
#y = np.array(y)
#y = to_categorical(y, num_classes=len(GENRES))

As you can see the audio sample jazz.00054.wav failed to be processed

In [10]:
#load spectrogram arrays only
#X, y = generate_save_spectrograms(GTZAN_DS, GENRES, GTZAN_NP_ARR, save_arr=True, load=False)
X, y = generate_save_spectrograms(GTZAN_DS, GENRES, GTZAN_NP_ARR, save_arr=False, load=True)

X_load = np.array(X)
y_load = np.array(y)
y_load = to_categorical(y, num_classes=len(GENRES))

In [11]:
X_load.shape

(999, 224, 224, 3)

In [12]:
X_load[:,:,:,0]

array([[[-18.630653, -36.445663, -30.848537, ..., -21.820604,
         -33.268562, -27.72534 ],
        [-11.96789 , -25.779314, -25.400127, ..., -17.889154,
         -27.190613, -22.919142],
        [ -7.065963, -17.002966, -20.253414, ..., -15.084603,
         -22.189814, -19.385345],
        ...,
        [-54.51265 , -59.36177 , -50.846977, ..., -53.9295  ,
         -51.788666, -47.09372 ],
        [-57.510082, -64.77551 , -56.344425, ..., -60.078465,
         -59.860992, -53.032635],
        [-60.70405 , -70.8052  , -62.980717, ..., -67.23171 ,
         -69.23209 , -59.996277]],

       [[-23.16937 , -32.50329 , -33.531204, ..., -26.820189,
         -34.92998 , -28.450407],
        [-18.801552, -22.509174, -25.487837, ..., -20.688082,
         -26.470762, -26.196243],
        [-15.355767, -14.814617, -18.043583, ..., -15.48569 ,
         -18.972654, -22.76359 ],
        ...,
        [-50.920395, -52.33997 , -46.292004, ..., -41.690544,
         -42.232502, -46.53174 ],
        [-56

In [13]:
X_load[:,:,:,1]

array([[[-18.630653, -36.445663, -30.848537, ..., -21.820604,
         -33.268562, -27.72534 ],
        [-11.96789 , -25.779314, -25.400127, ..., -17.889154,
         -27.190613, -22.919142],
        [ -7.065963, -17.002966, -20.253414, ..., -15.084603,
         -22.189814, -19.385345],
        ...,
        [-54.51265 , -59.36177 , -50.846977, ..., -53.9295  ,
         -51.788666, -47.09372 ],
        [-57.510082, -64.77551 , -56.344425, ..., -60.078465,
         -59.860992, -53.032635],
        [-60.70405 , -70.8052  , -62.980717, ..., -67.23171 ,
         -69.23209 , -59.996277]],

       [[-23.16937 , -32.50329 , -33.531204, ..., -26.820189,
         -34.92998 , -28.450407],
        [-18.801552, -22.509174, -25.487837, ..., -20.688082,
         -26.470762, -26.196243],
        [-15.355767, -14.814617, -18.043583, ..., -15.48569 ,
         -18.972654, -22.76359 ],
        ...,
        [-50.920395, -52.33997 , -46.292004, ..., -41.690544,
         -42.232502, -46.53174 ],
        [-56

In [14]:
X_load[:,:,:,2]

array([[[-18.630653, -36.445663, -30.848537, ..., -21.820604,
         -33.268562, -27.72534 ],
        [-11.96789 , -25.779314, -25.400127, ..., -17.889154,
         -27.190613, -22.919142],
        [ -7.065963, -17.002966, -20.253414, ..., -15.084603,
         -22.189814, -19.385345],
        ...,
        [-54.51265 , -59.36177 , -50.846977, ..., -53.9295  ,
         -51.788666, -47.09372 ],
        [-57.510082, -64.77551 , -56.344425, ..., -60.078465,
         -59.860992, -53.032635],
        [-60.70405 , -70.8052  , -62.980717, ..., -67.23171 ,
         -69.23209 , -59.996277]],

       [[-23.16937 , -32.50329 , -33.531204, ..., -26.820189,
         -34.92998 , -28.450407],
        [-18.801552, -22.509174, -25.487837, ..., -20.688082,
         -26.470762, -26.196243],
        [-15.355767, -14.814617, -18.043583, ..., -15.48569 ,
         -18.972654, -22.76359 ],
        ...,
        [-50.920395, -52.33997 , -46.292004, ..., -41.690544,
         -42.232502, -46.53174 ],
        [-56

I compared both arrays the ones generated during spectrogram creation and spectrogram load

As I am goin to work with the same data it is better to just load the arrays than creating the spectrograms each time

In [15]:
#np.array_equal(X, X_load)

In [16]:
y_load.shape

(999, 10)

In [17]:
#create splits using 80,20,20 as train,validation,test respectively

#split to 80% train and 20% test
X_train_f, X_test, y_train_f, y_test = train_test_split(
    X_load, y_load, test_size=0.2, stratify=y_load, random_state=42
)

#split the 80% train to 60% train and 20% validation
X_train, X_vldtn, y_train, y_vldtn = train_test_split(
    X_train_f, y_train_f, test_size=0.25, 
    stratify=y_train_f, random_state=42
)

print(len(X_train))
print(len(X_vldtn))
print(len(X_test))

599
200
200


I used stratify just to ensure class distribution is preserved in each set

#### Using MobileNetV2

Recall that I am using a pre-trained model: MobileNetV2

From the lectures and previous homework where tensorflow was introduced what I want to do is keep already trained convolutional layers which is called Transfer Learning

Now what I have to do is train a custom base model considering:
  * Use imagenet weights
  * Load pre-trained MobileNetV2 including only convolutional layers not dense layers
  * The input shape which is (224,224,3)

In [18]:
IN_SHAPE=X_load[0].shape
IN_SHAPE

(224, 224, 3)

#### Transfer Learning

I am trying to use my notes from previous homework so I can try to reproduce the sequential approach where each layer is added after the other instead of just using the function that includes all in one go

In [ ]:
#train a custom base model using MobileNetV2
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=IN_SHAPE
)

In [ ]:
#'freeze' pre-trained model's weights during training
base_model.trainable=False

In [ ]:
inputs = keras.Input(shape=IN_SHAPE)
outputs = base_model(inputs)
model = keras.Model(inputs, outputs)

Pre-process the input

In [19]:
#preprocess input for MobileNetV2
X_train_pre = preprocess_input(X_train)
X_vldtn_pre = preprocess_input(X_vldtn)

In [20]:
X_train_pre.shape

(599, 224, 224, 3)

I got a dependency error from cuda toolkit so I had to update my cuda toolkit version as the one I had was 11.5 and I needed at least 11.8

In case you need it: [cuda toolkit installation guide](https://docs.nvidia.com/cuda/cuda-installation-guide-linux/)

In [ ]:
predictions = model.predict(X_train_pre)
predictions.shape

So here I have a 3D output 

I have 7x7x1280 3D so I want each 7x7 to be 1D

#### Implementation comments

One thing to consider for spectrogram data is that Rectangular-shaped pooling regions are preferred because they align with the aspect ratios of rectangular convolutional kernels used in spectrogram analysis

This allows for more wider regions to capture longer temporal features while taller regions preserve finer frequency details, since temporal features often span longer durations

For now I am starting out with a square pooling region such as GlobalAveragePooling which can be considered a subset in case a Generalized Mean Pooling is implemented

Note: I did not specify a BatchLoader so this looks a bit different from the lecture or homework, I am doing that later

#### Pooling

In [ ]:
inputs = keras.Input(shape=IN_SHAPE)
base = base_model(inputs)
outputs = GlobalAveragePooling2D()(base)
model = keras.Model(inputs, outputs)

In [ ]:
predictions = model.predict(X_train_pre)
predictions.shape

#### Dense Layers

Now with the dimensionality reduction drom 3D -> 1D it is time to turn the 1D vectors into predictions, so that 1D vector representation is what is turned into a prediction considering the number of classes (the number of genres in this case) for now it is just the base model not the actual training

In [ ]:
# (599, 1280) => (599, 10) where 10 is the number of genres or classes
inputs = keras.Input(shape=IN_SHAPE)
base = base_model(inputs, training=False)
vectors = GlobalAveragePooling2D()(base)
outputs = Dense(len(GENRES))(vectors)
model = keras.Model(inputs, outputs)

In [ ]:
predictions = model.predict(X_train_pre)
predictions.shape

#### Optimizer

Remember that weights are changed in such a way that it tries to find a better solution or in other words tries to do a more accurate prediction or approximation

The recommended optimizer is Adam and it was used in the lectures, so I have to:
  * Start with a learning rate e.g. 0.01 and adjust it later or use different learning rates and select the best one
  * Use the appropriate loss function as this is a multi-classification task such as CategoricalCrossEntropy as there are two or more label classes

At this point it is better to pack previous model steps in a function to identify suitable parameters, in this case learning rate

In [ ]:
METRIC="accuracy"
scores = {}
learn_rates = [0.0001, 0.001, 0.01, 0.1]

In [ ]:
def make_model(learning_rate):
    base_model = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=IN_SHAPE
    )
    
    base_model.trainable = False

    inputs = keras.Input(shape=IN_SHAPE)
    base = base_model(inputs, training=False)
    vectors = GlobalAveragePooling2D()(base)
    outputs = Dense(len(GENRES))(vectors)
    model = keras.Model(inputs, outputs)

    optimizer = Adam(learning_rate=learning_rate)
    loss = CategoricalCrossentropy(from_logits=True)
    
    model.compile(
        optimizer=optimizer, 
        loss=loss, 
        metrics=[METRIC]
    )

    return model

Testing learning rates

In [ ]:
for lr in learn_rates:
    print(f"Learning rate: {lr}")

    model = make_model(learning_rate=lr)
    history = model.fit(X_train_pre, y_train, epochs=50, batch_size=32, validation_data=(X_vldtn_pre, y_vldtn))
    scores[lr] = history.history

    print("====")

Visualize the behaviour of each learning rate value

In [ ]:
for lr, history in scores.items():
    plt.plot(history["accuracy"], label=lr)

plt.xticks(np.arange(50), rotation=90)
plt.title("Training accuracy")
plt.legend()
plt.show()

In [ ]:
for lr, history in scores.items():
    plt.plot(history["val_accuracy"], label=lr)

plt.xticks(np.arange(50), rotation=90)
plt.title("Validation accuracy")
plt.legend()
plt.show()

From the visualizations the best learning rate for validation accuracy is 0.01

#### Add more layers

Adding more layers can help in achieving higher performance if possible or course

I added an inner layer

In [ ]:
LEARN_RT=0.01
ACTIVATE="relu"

In [ ]:
def make_model(size_inner):
    base_model = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=IN_SHAPE
    )
    
    base_model.trainable = False

    inputs = keras.Input(shape=IN_SHAPE)
    base = base_model(inputs, training=False)
    vectors = GlobalAveragePooling2D()(base)

    inner_layer = Dense(size_inner, activation=ACTIVATE)(vectors)

    outputs = Dense(len(GENRES))(inner_layer)
    model = keras.Model(inputs, outputs)

    optimizer = Adam(learning_rate=LEARN_RT)
    loss = CategoricalCrossentropy(from_logits=True)
    
    model.compile(
        optimizer=optimizer, 
        loss=loss, 
        metrics=[METRIC]
    )

    return model

In [ ]:
scores = {}
inner_sizes = [10, 100, 1000]

for sz in inner_sizes:
    print(f"Inner size: {sz}")

    model = make_model(size_inner=sz)
    history = model.fit(X_train_pre, y_train, epochs=50, batch_size=32, validation_data=(X_vldtn_pre, y_vldtn))
    scores[sz] = history.history

    print("====")

In [ ]:
for sz, history in scores.items():
    plt.plot(history["accuracy"], label=sz)

plt.xticks(np.arange(50), rotation=90)
plt.title("Training accuracy")
plt.legend()
plt.show()

In [ ]:
for sz, history in scores.items():
    plt.plot(history["val_accuracy"], label=sz)

plt.xticks(np.arange(50), rotation=90)
#plt.ylim(0.40, 0.60)
plt.title("Validation accuracy")
plt.legend()
plt.show()

In [ ]:
INN_SZ = 1000

#### Droput and Regularization

Freezing a certain part to apply regularization so overfitting is mitigated at some extent

In [ ]:
def make_model(droprate):
    base_model = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=IN_SHAPE
    )
    
    base_model.trainable = False

    inputs = keras.Input(shape=IN_SHAPE)
    base = base_model(inputs, training=False)
    vectors = GlobalAveragePooling2D()(base)

    inner_layer = Dense(INN_SZ, activation=ACTIVATE)(vectors)
    drop = Dropout(rate=droprate)(inner_layer)

    outputs = Dense(len(GENRES))(drop)
    model = keras.Model(inputs, outputs)

    optimizer = Adam(learning_rate=LEARN_RT)
    loss = CategoricalCrossentropy(from_logits=True)
    
    model.compile(
        optimizer=optimizer, 
        loss=loss, 
        metrics=[METRIC]
    )

    return model

In [ ]:
scores = {}
drop_rates = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7]

for dr in drop_rates:
    print(f"Drop rate: {dr}")

    model = make_model(
        droprate=dr
    )

    model = make_model(droprate=dr)
    history = model.fit(X_train_pre, y_train, epochs=50, batch_size=32, validation_data=(X_vldtn_pre, y_vldtn))
    scores[dr] = history.history

    print("====")

In [ ]:
for dr, history in scores.items():
    plt.plot(history["val_accuracy"], label=dr)

plt.title("Validation accuracy")
plt.legend()
plt.show()

In [ ]:
hist = scores[0.1]
plt.plot(hist["accuracy"], label="acc")
plt.plot(hist["val_accuracy"], label="val_acc")
plt.legend()
plt.show()

In [ ]:
hist = scores[0.2]
plt.plot(hist["accuracy"], label="acc")
plt.plot(hist["val_accuracy"], label="val_acc")
plt.legend()
plt.show()

In [ ]:
DROP = 0.1

#### Train a larger model

After testing for few epochs and finding out the parameters I can try more epochs and store a model

In [ ]:
def make_model():
    base_model = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=IN_SHAPE
    )
    
    base_model.trainable = False

    inputs = keras.Input(shape=IN_SHAPE)
    base = base_model(inputs, training=False)
    vectors = GlobalAveragePooling2D()(base)

    inner_layer = Dense(INN_SZ, activation=ACTIVATE)(vectors)
    drop = Dropout(rate=DROP)(inner_layer)

    outputs = Dense(len(GENRES))(drop)
    model = keras.Model(inputs, outputs)

    optimizer = Adam(learning_rate=LEARN_RT)
    loss = CategoricalCrossentropy(from_logits=True)
    
    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=[METRIC]
    )

    return model

Model checkoint so I can store best model as epochs are completed

In [ ]:
checkpoint = ModelCheckpoint(
    "../models/mobilenetv2_v2_{epoch:02d}_val_acc_{val_accuracy:.3f}.keras",
    save_best_only=True,
    monitor="val_accuracy",
    mode="max"
)

In [ ]:
EPOCHS=50
BATCH=32

model = make_model()
history = model.fit(
    X_train_pre, 
    y_train, 
    epochs=EPOCHS, 
    batch_size=BATCH, 
    validation_data=(X_vldtn_pre, y_vldtn),
    callbacks=[checkpoint]
)

After trying different windows i reached a 0.675 validation accuracy

#### Implementation comments

After some reading I found out that pooling methods that account for the distinct characteristics of the time and frequency dimensions are recommended over standard square pooling which was my first approach.

Rectangular-shaped regions are preferred because they align with the aspect ratios of rectangular convolutional kernels used in spectrogram analysis, allowing for more precise modeling of time-frequency patterns.

This approach enables wider pooling regions to capture longer temporal features while taller regions preserve finer frequency details since temporal features often span longer durations.

Additionally asymmetric pooling strategies that incorporate different downsampling techniques for time and frequency have been proposed to enhance robustness to time shifts and sensitivity to frequency shifts. Methods such as Generalized Mean Pooling (GeMPool) and AutoPool are recommended as they allow the model to learn optimal aggregation strategies though trainable parameters, offering flexibility between average and max pooling hehaviour.

#### Average and Max Pooling

To keep things simple it seems reasonable to dynamically balance between average and max pooling during training. There are two approaches apply AutoPool along the time axis (to aggregate temporal features) or across frequency bins

  * *Along time axis*: Which moments in time are most important? Common in classification tasks where *temporal context* matters

  * *Across frequency bins*: Which frequency bands are most relevant? Helps in pitch detection or source separation

As the task is classification I can try along the time axis first and then to compare apply across frequency bins or use both sequentially (this is optional)


#### AutoPool considerations

Applying to time axis

  * Base model outputs (batch, height, width, channels) have to reshape to 3D for AutoPool input
  * Autopool can be more stable using non-negative values
  * Can use regularization to prevent overfitting (L2) may need to be tuned